In [1]:
# nb01_fetch_raw.py
import yfinance as yf
import pandas as pd
import numpy as np
import os
from google.colab import drive

drive.mount("/content/drive")

PROJECT_DIR = "/content/drive/MyDrive/volatility-forecast"
RAW_DIR = os.path.join(PROJECT_DIR, "data", "raw")

if not os.path.isdir(PROJECT_DIR):
    PROJECT_DIR = "/content/volatility-forecast"
    RAW_DIR = os.path.join(PROJECT_DIR, "data", "raw")
    print("Drive not found — falling back to /content")

os.makedirs(RAW_DIR, exist_ok=True)
print("RAW_DIR =", RAW_DIR)

# ticker : QQQ(target) + 8 macro X
TICKERS = ["QQQ", "^VIX", "^TNX", "^IRX", "HYG", "LQD", "TLT", "GLD"]
START, END = "2011-01-01", "2026-05-29"

# download — auto_adjust=True (price assets→adjusted close; indices unaffected)
raw = yf.download(TICKERS, start=START, end=END,
                  auto_adjust=True, group_by="ticker", progress=False)

# extract Close per ticker → outer join (NO dropna, NO fill — raw stays raw)
close = pd.concat({t: raw[t]["Close"] for t in TICKERS}, axis=1)
close.index.name = "Date"
print("shape:", close.shape)

audit = pd.DataFrame({
    "first_valid": close.apply(lambda s: s.first_valid_index()),
    "last_valid":  close.apply(lambda s: s.last_valid_index()),
    "n_nan":       close.isna().sum(),
    "n_rows":      len(close),
})
print(audit)

out_path = os.path.join(RAW_DIR, "raw_close.csv")
close.to_csv(out_path)
print("saved →", out_path)

Mounted at /content/drive
RAW_DIR = /content/drive/MyDrive/volatility-forecast/data/raw
shape: (3874, 8)
     first_valid last_valid  n_nan  n_rows
QQQ   2011-01-03 2026-05-28      1    3874
^VIX  2011-01-03 2026-05-28      0    3874
^TNX  2011-01-03 2026-05-28      2    3874
^IRX  2011-01-03 2026-05-28      2    3874
HYG   2011-01-03 2026-05-28      1    3874
LQD   2011-01-03 2026-05-28      1    3874
TLT   2011-01-03 2026-05-28      1    3874
GLD   2011-01-03 2026-05-28      1    3874
saved → /content/drive/MyDrive/volatility-forecast/data/raw/raw_close.csv
